In [2]:
import pandas
df = pd.read_csv('Results.csv')

<IPython.core.display.Javascript object>

In [3]:
df1 =pd.read_csv('polling.csv')

<IPython.core.display.Javascript object>

In [5]:
df1.columns

Index(['Part  No', 'Polling  Station  No.',
       'Location  and  Name  of Building  in  Which  Polling Station  Located',
       'Polling  Areas*',
       'Whether for  all voters or  men only or Women only'],
      dtype='object')

In [6]:
merged_df = pd.merge(df, df1, on='Polling  Station  No.', how='inner')

<IPython.core.display.Javascript object>

In [7]:
merged_df.to_csv('All.csv')

In [10]:
df =pd.read_csv('All.csv')

<IPython.core.display.Javascript object>

In [11]:
df.columns

Index(['SL.  NO.', 'Polling  Station  No.', 'Naam  Tamilar Katchi',
       'Dravida Munnetra Kazhagam',
       'All  India  Anna Dravida Munnetra Kazhagam',
       'All  India\nPuratchi Thalaivar Makkal Munnettra', 'Puthiya Tamilagam',
       'Veerath\nThiyagi Viswanathados s Thozhilalarkal',
       'Tamilaga Vettri Kazhagam', 'Independent', 'Independent.1',
       'Independent.2', 'Independent.3', 'Independent.4', 'Independent.5',
       'Independent.6', 'Independent.7', 'Independent.8', 'Independent.9',
       'Independent.10', 'Independent.11', 'Independent.12', 'Independent.13',
       'Independent.14', 'Independent.15', 'Independent.16', 'Independent.17',
       'Independent.18', 'Total  of  valid  votes', 'No.  of  Rejected  Votes',
       'Votes  for  'NOTA'', 'Total', 'No.  of  Tendered  votes', 'Part  No',
       'Location  and  Name  of Building  in  Which  Polling Station  Located',
       'Polling  Areas*',
       'Whether for  all voters or  men only or Women only'],
     

In [15]:
import os
import textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill

# ==============================================================================
# STEP 1: DEFINE SYSTEM CONFIGURATION (EXACT COLUMNS FROM NEW DATASET)
# ==============================================================================
party_config = {
    'Dravida Munnetra Kazhagam': {'label': 'DMK', 'color': '#D32F2F'},
    'All India Anna Dravida Munnetra Kazhagam': {'label': 'AIADMK', 'color': '#2E7D32'},
    'Tamilaga Vettri Kazhagam': {'label': 'TVK', 'color': '#7B1E1E'},
    'Naam Tamilar Katchi': {'label': 'NTK', 'color': '#FBC02D'},
    'Puthiya Tamilagam': {'label': 'PT', 'color': '#FF5722'},
    'All India Puratchi Thalaivar Makkal Munnettra': {'label': 'AIPTMMK', 'color': '#607D8B'},
    'Veerath Thiyagi Viswanathados s Thozhilalarkal': {'label': 'VTVT', 'color': '#9C27B0'}
}

party_columns = list(party_config.keys())

output_dir = 'branded_polling_charts'
os.makedirs(output_dir, exist_ok=True)
excel_filename = 'Branded_Polling_Station_Report.xlsx'

# ==============================================================================
# WORKSPACE CLEANUP & INITIALIZATION (FIXED STR ACCESSOR CRASH)
# ==============================================================================
print("Clearing cache layers, dropping untitled columns, and resetting DataFrame...")

# Load your file here
df = pd.read_csv("All.csv")

# FIX: Pure python sequence transformation handles complex column cleaning reliably
cleaned_columns = []
for col in df.columns:
    col_str = str(col).replace('\n', ' ').strip()          # Clear line breaks
    col_str = " ".join(col_str.split())                   # Clear wide whitespace gaps
    # Safe match patterns for NOTA variants containing internal single quotes
    if "NOTA" in col_str or "Votes for 'NOTA'" in col_str:
        col_str = "NOTA"
    cleaned_columns.append(col_str)

df.columns = cleaned_columns

columns_to_drop = [f'{col}_Rank' for col in party_columns] + [
    'Winner_Party', 'Winner_Votes', 'Runner_Up_Votes', 
    'Margin_Of_Victory', 'Runner_Up_Party', 'Margin_Percentage', 'Total_Other_Candidate_Votes',
    'Unnamed: 0', 'Total_Independent_Votes'
]
df = df.drop(columns=[c for c in columns_to_drop if c in df.columns], errors='ignore')

# Exact single-spaced structural variables matched to your dataset rules
station_col = 'Polling Station No.'
locality_col = 'Location and Name of Building in Which Polling Station Located'
area_col = 'Polling Areas*'

df[station_col] = pd.to_numeric(df[station_col], errors='coerce').fillna(0).astype(int)

# Group structural columns to isolate the 19 Independent candidate entries safely
non_independent_cols = party_columns + [
    'SL. NO.', station_col, locality_col, area_col, 'Total of valid votes', 
    'No. of Rejected Votes', 'NOTA', 'Total', 'No. of Tendered votes', 'Part No', 
    'Whether for all voters or men only or Women only'
]
independent_cols = [c for c in df.columns if c not in non_independent_cols]
df[independent_cols] = df[independent_cols].fillna(0)

# ==============================================================================
# STEP 2: PRIMARY VOTING CALCULATIONS (WINNER, MARGIN, AND RANKINGS)
# ==============================================================================
print("Running vote analytics calculations...")

df['Winner_Party'] = df[party_columns].idxmax(axis=1)
df['Winner_Votes'] = df[party_columns].max(axis=1)

sorted_votes = np.sort(df[party_columns].values, axis=1)
df['Runner_Up_Votes'] = sorted_votes[:, -2]
df['Margin_Of_Victory'] = df['Winner_Votes'] - df['Runner_Up_Votes']

sorted_indices = np.argsort(df[party_columns].values, axis=1)
df['Runner_Up_Party'] = [party_columns[idx[-2]] for idx in sorted_indices]

ranks_df = df[party_columns].rank(axis=1, ascending=False, method='min').astype(int)
ranks_df = ranks_df.rename(columns={col: f'{col}_Rank' for col in party_columns})
df = pd.concat([df, ranks_df], axis=1)

# ==============================================================================
# STEP 3: ADVANCED STRATEGIC METRICS (SWINGS, DOMINANCE, AND SPOILERS)
# ==============================================================================
print("Extracting strategic election metrics...")

df['Margin_Percentage'] = np.where(
    df['Total of valid votes'] > 0, 
    (df['Margin_Of_Victory'] / df['Total of valid votes']) * 100, 
    0
)

critical_swings = df[df['Margin_Percentage'] <= 5.0][
    [station_col, locality_col, area_col, 'Winner_Party', 'Runner_Up_Party', 'Margin_Of_Victory', 'Margin_Percentage']
].sort_values(by=station_col, ascending=True)

locality_counts = df.groupby([locality_col, 'Winner_Party']).size().unstack(fill_value=0)
locality_dominance_summary = pd.DataFrame({
    'Total_Booths_In_Locality': df.groupby(locality_col).size(),
    'Dominant_Party': locality_counts.idxmax(axis=1) if not locality_counts.empty else None,
    'Dominant_Party_Booths_Won': locality_counts.max(axis=1) if not locality_counts.empty else 0
}).reset_index()

if independent_cols:
    df['Total_Independent_Votes'] = df[independent_cols].sum(axis=1)
    independent_vote_splitting = df.groupby(locality_col).agg(
        Total_Valid_Votes_In_Locality=('Total of valid votes', 'sum'),
        Total_Independent_Votes_In_Locality=('Total_Independent_Votes', 'sum')
    ).reset_index()
    independent_vote_splitting['Independent_Vote_Share_%'] = np.where(
        independent_vote_splitting['Total_Valid_Votes_In_Locality'] > 0,
        (independent_vote_splitting['Total_Independent_Votes_In_Locality'] / independent_vote_splitting['Total_Valid_Votes_In_Locality']) * 100,
        0
    )
    independent_vote_splitting = independent_vote_splitting.sort_values(by='Independent_Vote_Share_%', ascending=False)

df = df.sort_values(by=station_col, ascending=True).reset_index(drop=True)

# ==============================================================================
# STEP 4: GENERATE MACRO CONSTITUENCY DASHBOARD CHART
# ==============================================================================
print("Compiling constituency macro-level summary dashboard...")
total_valid_constituency_votes = df['Total of valid votes'].sum()

macro_votes = df[party_columns].sum()
macro_percentages = (macro_votes / total_valid_constituency_votes) * 100

summary_df = pd.DataFrame({
    'Raw_Column': party_columns,
    'Label': [party_config[col]['label'] for col in party_columns],
    'Color': [party_config[col]['color'] for col in party_columns],
    'Total_Votes': macro_votes.values,
    'Share_Percentage': macro_percentages.values
}).sort_values(by='Total_Votes', ascending=True)

fig, ax = plt.subplots(figsize=(13, 7))
macro_bars = ax.barh(summary_df['Label'], summary_df['Share_Percentage'], color=summary_df['Color'], edgecolor='black', height=0.7)
ax.invert_yaxis()

for bar, pct, votes in zip(macro_bars, summary_df['Share_Percentage'], summary_df['Total_Votes']):
    width = bar.get_width()
    ax.text(width + (summary_df['Share_Percentage'].max() * 0.015), bar.get_y() + bar.get_height()/2, f"{int(votes):,} Votes ({pct:.2f}%)", 
            va='center', ha='left', fontsize=9, fontweight='bold')

ax.set_title(f"CONSTITUENCY MACRO SUMMARY DASHBOARD\nTotal Valid Votes Cast: {int(total_valid_constituency_votes):,}", fontsize=13, fontweight='bold', pad=20)
ax.set_xlabel('Overall Vote Share Percentage (%)', fontsize=11)
ax.set_xlim(0, summary_df['Share_Percentage'].max() * 1.25)
plt.tight_layout()

macro_chart_path = f"{output_dir}/Constituency_Macro_Summary.png"
plt.savefig(macro_chart_path, dpi=130, bbox_inches='tight')
plt.close()

# ==============================================================================
# STEP 5: INDIVIDUAL BOOTH VISUALS
# ==============================================================================
print("Generating dynamically sorted booth images...")
for index, row in df.iterrows():
    station_id = str(int(row[station_col]))
    locality_name = str(row[locality_col]).strip()
    area_name = str(row[area_col]).strip()
    total_votes = row['Total of valid votes']
    
    if pd.isna(total_votes) or total_votes == 0:
        continue
        
    station_data = pd.DataFrame({
        'Label': [party_config[col]['label'] for col in party_columns],
        'Color': [party_config[col]['color'] for col in party_columns],
        'Votes': row[party_columns].values.astype(float)
    })
    station_data['Percentage'] = (station_data['Votes'] / total_votes) * 100
    station_data = station_data.sort_values(by='Votes', ascending=True)
    
    fig, ax = plt.subplots(figsize=(11, 6.5))
    bars = ax.barh(station_data['Label'], station_data['Percentage'], color=station_data['Color'], edgecolor='black', height=0.6)
    ax.invert_yaxis()
    
    for bar, pct, count in zip(bars, station_data['Percentage'], station_data['Votes']):
        width = bar.get_width()
        ax.text(width + (station_data['Percentage'].max() * 0.015), bar.get_y() + bar.get_height()/2, f"{int(count)} Votes ({pct:.1f}%)", 
                va='center', ha='left', fontsize=8, fontweight='bold')
                
    raw_title_text = f"Station {station_id} | Locality: {locality_name}\nArea: {area_name}"
    wrapped_title = "\n".join(textwrap.wrap(raw_title_text, width=75))
    ax.set_title(f"{wrapped_title} (Total: {int(total_votes)})", fontsize=10, fontweight='bold', pad=15)
    ax.set_xlabel('Vote Share Percentage (%)', fontsize=9)
    ax.set_xlim(0, min(100, station_data['Percentage'].max() * 1.35))  
    plt.tight_layout()
    
    safe_locality = "".join([c for c in locality_name if c.isalnum() or c in (' ', '_', '-')]).strip().replace(' ', '_')
    plt.savefig(f"{output_dir}/Station_{station_id}_{safe_locality}.png", dpi=90, bbox_inches='tight')
    plt.close()

# ==============================================================================
# STEP 6: EXPORT TO CLEAN MULTI-SHEET EXCEL WORKBOOK
# ==============================================================================
print("Assembling structured spreadsheet file panels (No Image Columns)...")
with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Detailed_Polling_Data', index=False)
    summary_df[['Label', 'Total_Votes', 'Share_Percentage']].sort_values(by='Total_Votes', ascending=False).to_excel(writer, sheet_name='Constituency_Dashboard', index=False)
    critical_swings.to_excel(writer, sheet_name='Critical_Swing_Booths', index=False)
    locality_dominance_summary.to_excel(writer, sheet_name='Locality_Dominance', index=False)
    
    # Conditional export safeguard for independent candidate analysis metrics
    if 'independent_vote_splitting' in locals() or 'independent_vote_splitting' in globals():
        independent_vote_splitting.to_excel(writer, sheet_name='Independent_Splitting', index=False)

# Re-load the freshly written workbook to safely inject assets via openpyxl drawing tools
wb = load_workbook(excel_filename)

# Safely overlay the macro dashboard summary plot onto the primary overview panel
if 'Constituency_Dashboard' in wb.sheetnames:
    ws_dash = wb['Constituency_Dashboard']
    if os.path.exists(macro_chart_path):
        img_macro = Image(macro_chart_path)
        ws_dash.add_image(img_macro, 'E2')

# ==============================================================================
# STEP 7: AUTO-FIT COLUMN WIDTHS & STYLING (FIXED FOR OPENPYXL CELLS)
# ==============================================================================
print("Optimizing Excel grid padding layout widths and styles...")
from openpyxl.styles import Font, PatternFill

for sheet in wb.worksheets:
    # 1. Format headers cleanly by targeting individual cell objects inside Row 1 only
    for cell in sheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    
    # 2. Fix layout tracking dimensions safely by referencing the cell's column index
    for col in sheet.columns:
        max_len = 0
        actual_column_index = col[0].column # Safely read the index integer property from the first cell object
        col_letter = get_column_letter(actual_column_index)
        
        for cell in col:
            if cell.value is not None:
                max_len = max(max_len, len(str(cell.value)))
        sheet.column_dimensions[col_letter].width = max(max_len + 3, 11)

wb.save(excel_filename)
print(f"\nSUCCESS! Branded workbook reporting environment finalized: '{excel_filename}'")


Clearing cache layers, dropping untitled columns, and resetting DataFrame...
Running vote analytics calculations...
Extracting strategic election metrics...
Compiling constituency macro-level summary dashboard...
Generating dynamically sorted booth images...
Assembling structured spreadsheet file panels (No Image Columns)...
Optimizing Excel grid padding layout widths and styles...

SUCCESS! Branded workbook reporting environment finalized: 'Branded_Polling_Station_Report.xlsx'


In [17]:
import pandas as pd
import os

# Input Excel file
excel_file = "viralimalai_results.xlsx"

# Output directory
output_dir = "csv_output"
os.makedirs(output_dir, exist_ok=True)

# Load Excel file
xls = pd.ExcelFile(excel_file)

print(f"Found sheets: {xls.sheet_names}")

# Iterate through each sheet
for sheet_name in xls.sheet_names:
    try:
        # Read sheet
        df = pd.read_excel(xls, sheet_name=sheet_name)

        # Clean sheet name (remove special chars for filename safety)
        safe_sheet_name = "".join(c if c.isalnum() else "_" for c in sheet_name)

        # Output file path
        output_file = os.path.join(output_dir, f"{safe_sheet_name}.csv")

        # Save as CSV
        df.to_csv(output_file, index=False)

        print(f"✅ Saved: {output_file}")

    except Exception as e:
        print(f"❌ Error processing sheet '{sheet_name}': {e}")

print("🎯 All sheets converted successfully!")

Found sheets: ['Detailed_Polling_Data', 'Constituency_Dashboard', 'Critical_Swing_Booths', 'Locality_Dominance', 'Independent_Splitting']
✅ Saved: csv_output/Detailed_Polling_Data.csv
✅ Saved: csv_output/Constituency_Dashboard.csv
✅ Saved: csv_output/Critical_Swing_Booths.csv
✅ Saved: csv_output/Locality_Dominance.csv
✅ Saved: csv_output/Independent_Splitting.csv
🎯 All sheets converted successfully!


In [19]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. LOAD DATASET AND STANDARDISE HEADERS
# ==============================================================================
# Update this string to match your source CSV file name
df7 = pd.read_csv("Detailed_Polling_Data.csv")

# Pure python sequence transformation handles complex column cleaning reliably
cleaned_columns = []
for col in df7.columns:
    col_str = str(col).replace('\n', ' ').strip()          # Clear line breaks
    col_str = " ".join(col_str.split())                   # Clear wide whitespace gaps
    # Safe match patterns for NOTA variants containing internal single quotes
    if "NOTA" in col_str or "Votes for 'NOTA'" in col_str:
        col_str = "NOTA"
    cleaned_columns.append(col_str)

df7.columns = cleaned_columns

# Wipe out any pre-existing calculated column copies to avoid value re-assignment errors
cols_to_clear = ['Margin_Percentage', 'Winner_Votes', 'Runner_Up_Votes', 'Margin_Of_Victory', 'Winner_Party', 'Cluster_ID']
df7 = df7.drop(columns=[c for c in cols_to_clear if c in df7.columns], errors='ignore')

# ==============================================================================
# 2. DEFINE DATASET CORE PARAMETERS
# ==============================================================================
core_parties = [
    'Dravida Munnetra Kazhagam',
    'All India Anna Dravida Munnetra Kazhagam', 
    'Tamilaga Vettri Kazhagam',
    'Naam Tamilar Katchi',
    'Puthiya Tamilagam',
    'All India Puratchi Thalaivar Makkal Munnettra',
    'Veerath Thiyagi Viswanathados s Thozhilalarkal'
]

# Exact structural column definitions from your current dataset layout
station_col = 'Polling Station No.'
building_col = 'Location and Name of Building in Which Polling Station Located'
area_col = 'Polling Areas*'

# Isolate all 19 unique independent column tracking fields dynamically
non_independent_cols = core_parties + [
    'SL. NO.', station_col, building_col, area_col, 'Total of valid votes', 
    'No. of Rejected Votes', 'NOTA', 'Total', 'No. of Tendered votes', 'Part No', 
    'Whether for all voters or men only or Women only'
]
independent_candidate_cols = [c for c in df7.columns if c not in non_independent_cols]

# CRITICAL FIX: Convert all party, independent, and NOTA columns to true numeric floats
# This safely converts stray text strings into NaN without causing mathematical sum crashes
all_data_numeric_cols = core_parties + independent_candidate_cols + ['NOTA']
for col in all_data_numeric_cols:
    if col in df7.columns:
        df7[col] = pd.to_numeric(df7[col], errors='coerce').fillna(0)

# ==============================================================================
# 3. CALCULATE NORMALIZED METRICS FOR THE MACHINE LEARNING MODEL
# ==============================================================================
# Calculate total independent votes dynamically across all 19 lanes safely now
df7['Total_Independent_Votes'] = df7[independent_candidate_cols].sum(axis=1)

# Calculate true total votes for normalization (Core Parties + 19 Independents + NOTA)
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['Total_Independent_Votes'] + df7['NOTA']

# Filter out empty entries to completely avoid division by zero errors
df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()

# Explicit Abbreviation Mapping to keep feature streams completely unique
party_abbreviations = {
    'Dravida Munnetra Kazhagam': 'DMK',
    'All India Anna Dravida Munnetra Kazhagam': 'AIADMK',
    'Tamilaga Vettri Kazhagam': 'TVK',
    'Naam Tamilar Katchi': 'NTK',
    'Puthiya Tamilagam': 'PT',
    'All India Puratchi Thalaivar Makkal Munnettra': 'AIPTMMK',
    'Veerath Thiyagi Viswanathados s Thozhilalarkal': 'VTVT'
}

# Feature Engineering: Create normalized percentage shares (%)
share_cols = []
for party in core_parties:
    party_label = party_abbreviations[party]
    col_name = f'{party_label}_share_pct'
    
    if col_name in df7.columns:
        df7 = df7.drop(columns=[col_name])
        
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

df7['independent_share_pct'] = (df7['Total_Independent_Votes'] / df7['Total_Calculated_Votes']) * 100

# Primary voting calculations (Winner, Runner-up, Margin)
df7['Winner_Votes'] = df7[core_parties].max(axis=1)
sorted_votes = np.sort(df7[core_parties].values, axis=1)
df7['Runner_Up_Votes'] = sorted_votes[:, -2]
df7['Margin_Of_Victory'] = df7['Winner_Votes'] - df7['Runner_Up_Votes']
df7['Winner_Party'] = df7[core_parties].idxmax(axis=1)
df7['Margin_Percentage'] = (df7['Margin_Of_Victory'] / df7['Total_Calculated_Votes']) * 100

# Set up clean target feature tracking array
feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']
X = df7[feature_cols].copy().fillna(0)

# ==============================================================================
# 4. SCALE FEATURES AND RUN K-MEANS CLUSTERING
# ==============================================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# ==============================================================================
# 5. PRINT THE PROFILE SUMMARY BREAKDOWNS
# ==============================================================================
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
print(df7.groupby('Cluster_ID')[feature_cols].mean().round(2))

print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# ==============================================================================
# 6. EXPORT STRATEGIC TARGET SHEETS FOR GROUND CAMPAIGN TEAMS
# ==============================================================================
cluster_names = {
    0: "Cluster_0_Target", 
    1: "Cluster_1_Target", 
    2: "Cluster_2_Target", 
    3: "Cluster_3_Target"
}

for cluster_num in range(optimal_k):
    target_cols = [station_col, building_col, area_col, 'Winner_Party', 'Margin_Percentage']
    valid_target_cols = [c for c in target_cols if c in df7.columns]
    
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][valid_target_cols]
    filename = f"Dataset_7_Cluster_{cluster_num}_{cluster_names[cluster_num]}.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated cleanly for all 4 clusters.")



--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            DMK_share_pct  AIADMK_share_pct  TVK_share_pct  NTK_share_pct  \
Cluster_ID                                                                  
0                   15.74             54.40          18.76           4.31   
1                   16.47             49.79          18.53           5.41   
2                   24.81             40.81          24.26           3.99   
3                   26.96             48.61           0.00           3.85   

            PT_share_pct  AIPTMMK_share_pct  VTVT_share_pct  \
Cluster_ID                                                    
0                   0.08               0.36            0.07   
1                   0.24               0.79            0.06   
2                   0.11               0.21            0.07   
3                   0.13               0.40           14.21   

            independent_share_pct  Margin_Percentage  
Cluster_ID                                